# T5 — Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (2019)

_Papers · Transformers & LLMs_

**Cast every NLP task — translation, classification, question answering — as text in → text out, and pre-train one encoder-decoder Transformer to do them all.**

---

This notebook is a practice scaffold for the **T5 — Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F, random
torch.manual_seed(0); random.seed(0)

# ===== shared vocabulary: specials + 3 task-prefix tokens + digits 0..9 =====
PAD, BOS, EOS = 0, 1, 2
REV, INC, LAST = 3, 4, 5          # task-prefix tokens (the text-to-text "which task")
DIG0 = 6; V = DIG0 + 10           # digits 0..9 -> ids 6..15
def dig(d): return DIG0 + d
N = 4                              # fixed source length (isolates the POSITION effect)

# ===== text-to-text data: every task is (source string -> target string) with a prefix =====
def make_example():
    task = random.choice(["rev", "inc", "last"])
    seq = [random.randint(0, 9) for _ in range(N)]
    if task == "rev":   src = [REV]  + [dig(x) for x in seq];        tgt = [dig(x) for x in reversed(seq)]
    elif task == "inc": src = [INC]  + [dig(x) for x in seq];        tgt = [dig((x + 1) % 10) for x in seq]
    else:               src = [LAST] + [dig(x) for x in seq];        tgt = [dig(seq[-1])]
    return [BOS] + src + [EOS], [BOS] + tgt + [EOS]
def pad(b):
    m = max(len(x) for x in b); return torch.tensor([x + [PAD] * (m - len(x)) for x in b])
def batch(bs=64):
    ex = [make_example() for _ in range(bs)]; return pad([s for s, t in ex]), pad([t for s, t in ex])

# ===== T5 simplified relative position bias: bucket(offset) -> per-head scalar, added to the logit =====
def rel_bucket(rp, num_buckets=16, max_distance=64):     # bidirectional bucketing (\S 2.1, simplified)
    nb = num_buckets // 2                                 # half "past" (offset<=0), half "future" (offset>0)
    rb = (rp > 0) * nb                                    # future offsets live in the second half
    rp = abs(rp); max_exact = nb // 2
    if rp < max_exact:                                    # small offsets: their own bucket
        val = rp
    else:                                                 # large offsets: merged logarithmically
        val = max_exact + int(math.log(max(rp, 1) / max_exact) / math.log(max_distance / max_exact) * (nb - max_exact))
        val = min(val, nb - 1)
    return rb + val

# ----- worked example: query at position 2, keys 0..5 -> buckets [2,1,0,9,10,11] -----
q = 2
print("buckets for query@2, keys 0..5:", [rel_bucket(k - q) for k in range(6)])   # [2, 1, 0, 9, 10, 11]

class RelPosBias(nn.Module):
    def __init__(self, h, num_buckets=16, max_distance=64):
        super().__init__(); self.nb, self.md = num_buckets, max_distance
        self.emb = nn.Embedding(num_buckets, h)          # ONE scalar per (bucket, head); shared across layers
    def forward(self, q_len, k_len, device):
        bk = torch.tensor([[rel_bucket(j - i, self.nb, self.md) for j in range(k_len)]
                           for i in range(q_len)], device=device)
        return self.emb(bk).permute(2, 0, 1).unsqueeze(0)    # (1, h, q_len, k_len) -> add to logits

# ===== multi-head attention (imported primitives), position bias added to the logit =====
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.h, self.dk = h, d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d, bias=False) for _ in range(4))
    def split(self, x):
        B, S, _ = x.shape; return x.view(B, S, self.h, self.dk).transpose(1, 2)
    def forward(self, xq, xkv, bias=None, mask=None):
        Q, K, Vv = self.split(self.Wq(xq)), self.split(self.Wk(xkv)), self.split(self.Wv(xkv))
        s = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)     # scaled dot-product logits
        if bias is not None: s = s + bias                    # \S 2.1: position scalar added to the logit
        if mask is not None: s = s.masked_fill(mask, float("-inf"))
        out = (F.softmax(s, dim=-1) @ Vv).transpose(1, 2).contiguous().view(xq.shape[0], xq.shape[1], self.h * self.dk)
        return self.Wo(out)

class FF(nn.Module):
    def __init__(self, d, ff):
        super().__init__(); self.n = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
    def forward(self, x): return self.n(x)

# ===== T5-style blocks: LayerNorm (no bias) on the INPUT of each sub-component, then residual =====
class EncoderBlock(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.a = MHA(d, h); self.f = FF(d, ff)
        self.n1 = nn.LayerNorm(d, bias=False); self.n2 = nn.LayerNorm(d, bias=False)
    def forward(self, x, bias):
        x = x + self.a(self.n1(x), self.n1(x), bias)         # self-attention + position bias
        return x + self.f(self.n2(x))                        # feed-forward
class DecoderBlock(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.sa = MHA(d, h); self.ca = MHA(d, h); self.f = FF(d, ff)
        self.n1 = nn.LayerNorm(d, bias=False); self.n2 = nn.LayerNorm(d, bias=False); self.n3 = nn.LayerNorm(d, bias=False)
    def forward(self, x, enc, sbias, cmask):
        x = x + self.sa(self.n1(x), self.n1(x), sbias, cmask)  # causal self-attention + position bias
        x = x + self.ca(self.n2(x), enc)                       # cross-attention into the encoder (no mask)
        return x + self.f(self.n3(x))

class TinyT5(nn.Module):
    def __init__(self, V, d=64, h=4, ff=128, L=2, use_relpos=True):
        super().__init__(); self.use = use_relpos
        self.emb = nn.Embedding(V, d)
        self.enc = nn.ModuleList([EncoderBlock(d, h, ff) for _ in range(L)])
        self.dec = nn.ModuleList([DecoderBlock(d, h, ff) for _ in range(L)])
        self.encbias = RelPosBias(h); self.decbias = RelPosBias(h)
        self.fn = nn.LayerNorm(d, bias=False); self.head = nn.Linear(d, V, bias=False)
    def forward(self, src, tgt):
        e = self.emb(src)
        eb = self.encbias(src.size(1), src.size(1), src.device) if self.use else None
        for b in self.enc: e = b(e, eb)
        d = self.emb(tgt); T = tgt.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt.device), 1)[None, None]
        db = self.decbias(T, T, tgt.device) if self.use else None
        for b in self.dec: d = b(d, e, db, causal)
        return self.head(self.fn(d))

def train(use_relpos, steps=2500, lr=2e-3):
    torch.manual_seed(0); random.seed(0)
    net = TinyT5(V, use_relpos=use_relpos); opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(steps):
        src, tgt = batch()
        logits = net(src, tgt[:, :-1])                       # teacher forcing: predict tgt shifted by 1
        loss = F.cross_entropy(logits.reshape(-1, V), tgt[:, 1:].reshape(-1), ignore_index=PAD)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def evaluate(net, n=300):
    random.seed(123); per = {"rev": [0, 0], "inc": [0, 0], "last": [0, 0]}; name = {REV: "rev", INC: "inc", LAST: "last"}; tot = 0
    for _ in range(n):
        src, tgt = make_example(); t = name[src[1]]; s = pad([src]); ys = [BOS]
        for _ in range(N + 2):                               # greedy decode
            o = net(s, torch.tensor([ys])); nx = int(o[0, -1].argmax()); ys.append(nx)
            if nx == EOS: break
        p = ys[1:]
        if p and p[-1] == EOS: p = p[:-1]
        ok = (p == tgt[1:-1]); per[t][1] += 1; per[t][0] += int(ok); tot += int(ok)
    return tot / n, {k: f"{v[0]}/{v[1]}" for k, v in per.items()}

print("\nONE model, THREE text-to-text tasks, WITH relative position bias:")
net = train(True);  acc, per = evaluate(net);  print(f"  seq-acc={acc:.3f}  per-task={per}")
print("ABLATION: same model, relative position bias OFF:")
net2 = train(False); acc2, per2 = evaluate(net2); print(f"  seq-acc={acc2:.3f}  per-task={per2}")
# Typical (our small run, not the paper's numbers):
#   WITH bias    -> seq-acc 1.000  (rev 101/101, inc 100/100, last 99/99)
#   WITHOUT bias -> seq-acc 0.150  (rev 3/101, inc 10/100, last 32/99) -- order-dependent tasks collapse.
# Exact numbers vary by hardware/seed; this is our small-scale run, not the paper's reported result.

## Visualize the data & results

_Can one tiny T5-style encoder-decoder learn three tasks at once when they are all cast as text-to-text — and does removing T5's relative position bias (ablation) destroy the order-dependent ones?_

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F, random
torch.manual_seed(0); random.seed(0)
PAD, BOS, EOS, REV, INC, LAST, DIG0 = 0, 1, 2, 3, 4, 5, 6
V = DIG0 + 10; N = 4
def dig(d): return DIG0 + d
def make_example():
    task = random.choice(["rev", "inc", "last"]); seq = [random.randint(0, 9) for _ in range(N)]
    if task == "rev":   src = [REV]  + [dig(x) for x in seq];  tgt = [dig(x) for x in reversed(seq)]
    elif task == "inc": src = [INC]  + [dig(x) for x in seq];  tgt = [dig((x + 1) % 10) for x in seq]
    else:               src = [LAST] + [dig(x) for x in seq];  tgt = [dig(seq[-1])]
    return [BOS] + src + [EOS], [BOS] + tgt + [EOS]
def pad(b):
    m = max(len(x) for x in b); return torch.tensor([x + [PAD] * (m - len(x)) for x in b])
def batch(bs=64):
    ex = [make_example() for _ in range(bs)]; return pad([s for s, t in ex]), pad([t for s, t in ex])
def rel_bucket(rp, nb=16, md=64):
    half = nb // 2; rb = (rp > 0) * half; rp = abs(rp); me = half // 2
    val = rp if rp < me else min(me + int(math.log(max(rp, 1) / me) / math.log(md / me) * (half - me)), half - 1)
    return rb + val
class RelPosBias(nn.Module):
    def __init__(self, h, nb=16, md=64):
        super().__init__(); self.nb, self.md = nb, md; self.emb = nn.Embedding(nb, h)
    def forward(self, q, k, dev):
        bk = torch.tensor([[rel_bucket(j - i, self.nb, self.md) for j in range(k)] for i in range(q)], device=dev)
        return self.emb(bk).permute(2, 0, 1).unsqueeze(0)
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__(); self.h, self.dk = h, d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d, bias=False) for _ in range(4))
    def split(self, x):
        B, S, _ = x.shape; return x.view(B, S, self.h, self.dk).transpose(1, 2)
    def forward(self, xq, xkv, bias=None, mask=None):
        Q, K, Vv = self.split(self.Wq(xq)), self.split(self.Wk(xkv)), self.split(self.Wv(xkv))
        s = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        if bias is not None: s = s + bias
        if mask is not None: s = s.masked_fill(mask, float("-inf"))
        return self.Wo((F.softmax(s, -1) @ Vv).transpose(1, 2).contiguous().view(xq.shape[0], xq.shape[1], self.h * self.dk))
class FF(nn.Module):
    def __init__(self, d, ff):
        super().__init__(); self.n = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
    def forward(self, x): return self.n(x)
class Enc(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.a = MHA(d, h); self.f = FF(d, ff)
        self.n1 = nn.LayerNorm(d, bias=False); self.n2 = nn.LayerNorm(d, bias=False)
    def forward(self, x, b): x = x + self.a(self.n1(x), self.n1(x), b); return x + self.f(self.n2(x))
class Dec(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__(); self.sa = MHA(d, h); self.ca = MHA(d, h); self.f = FF(d, ff)
        self.n1 = nn.LayerNorm(d, bias=False); self.n2 = nn.LayerNorm(d, bias=False); self.n3 = nn.LayerNorm(d, bias=False)
    def forward(self, x, e, sb, cm):
        x = x + self.sa(self.n1(x), self.n1(x), sb, cm); x = x + self.ca(self.n2(x), e); return x + self.f(self.n3(x))
class TinyT5(nn.Module):
    def __init__(self, V, d=64, h=4, ff=128, L=2, use=True):
        super().__init__(); self.use = use; self.emb = nn.Embedding(V, d)
        self.enc = nn.ModuleList([Enc(d, h, ff) for _ in range(L)]); self.dec = nn.ModuleList([Dec(d, h, ff) for _ in range(L)])
        self.eb = RelPosBias(h); self.db = RelPosBias(h); self.fn = nn.LayerNorm(d, bias=False); self.head = nn.Linear(d, V, bias=False)
    def forward(self, src, tgt):
        e = self.emb(src); eb = self.eb(src.size(1), src.size(1), src.device) if self.use else None
        for b in self.enc: e = b(e, eb)
        d = self.emb(tgt); T = tgt.size(1)
        cm = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt.device), 1)[None, None]
        db = self.db(T, T, tgt.device) if self.use else None
        for b in self.dec: d = b(d, e, db, cm)
        return self.head(self.fn(d))
@torch.no_grad()
def quick(net, n=150):
    random.seed(7); tot = 0
    for _ in range(n):
        src, tgt = make_example(); s = pad([src]); ys = [BOS]
        for _ in range(N + 2):
            o = net(s, torch.tensor([ys])); nx = int(o[0, -1].argmax()); ys.append(nx)
            if nx == EOS: break
        p = ys[1:]
        if p and p[-1] == EOS: p = p[:-1]
        tot += int(p == tgt[1:-1])
    return round(tot / n, 3)
def track(use, evals=(0,50,100,150,200,300,400,600,800,1200,1600,2000,2500), lr=2e-3):
    torch.manual_seed(0); random.seed(0)
    net = TinyT5(V, use=use); opt = torch.optim.Adam(net.parameters(), lr=lr); es = set(evals); curve = []
    for st in range(max(evals) + 1):
        if st in es: curve.append([st, quick(net)])
        src, tgt = batch(); lg = net(src, tgt[:, :-1])
        loss = F.cross_entropy(lg.reshape(-1, V), tgt[:, 1:].reshape(-1), ignore_index=PAD)
        opt.zero_grad(); loss.backward(); opt.step()
    return curve
print("rel-pos ON :", track(True))
print("rel-pos OFF:", track(False))
# ON  -> 0.0, 0.36, 0.73, 0.96, ... 1.0 by ~step 300 (one model learns all three text-to-text tasks)
# OFF -> flat ~0.15-0.22 (order-dependent rev/inc collapse; only order-free <last> partly survives)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your one tiny T5 reaches ~100% on all three text-to-text tasks
            (<rev>, <inc>, <last>) with
            the relative position bias. You delete the position bias (the only thing the model knows
            about order) and retrain everything else identically. What happens to each task, and what
            does the split tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one thing: set use_relpos=False so no scalar is added to any attention logit. Keep depth, width, heads, optimizer, data, and seed identical. — _An honest ablation varies only the relative position bias, so any change is attributable to it._
- Retrain and read per-task accuracy: <rev> and <inc> collapse toward chance, while <last> only partly drops. — _Reversing and per-position +1 both require knowing each token's place; outputting the last symbol needs only "the end token," which survives weak position information._
- Conclude that the relative position bias — not extra capacity — is what lets the order-blind attention respect token order. — _Both runs share architecture and parameter count except for the position scalars, isolating them as the cause._

**Answer:** With the relative position bias removed, the order-dependent tasks
                 (<rev>, <inc>) collapse toward chance because plain
                 self-attention is permutation-invariant — it can see which digits are present but
                 not where they go. The <last> task degrades far less, since it
                 needs almost no ordering. In our run overall sequence accuracy fell from ~1.0 to ~0.15.
                 Because the two runs are identical except for the position scalars, this isolates the
                 relative position bias as the source of order information. The CODEVIZ panel shows the
                 contrast.

</details>

**Problem 2.** In the worked example (16 buckets, bidirectional, query at position 2), the keys at
            positions $[0,1,2,3,4,5]$ mapped to buckets $[2,1,0,9,10,11]$. Why do the left neighbour
            (offset $-1$) and right neighbour (offset $+1$) get different buckets (1 vs 9), and
            why does that matter for the reverse task?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the bucketing: bidirectional splits the 16 buckets into a "past" half (offsets $\le 0$, buckets 0–7) and a "future" half (offsets $\gt 0$, buckets 8–15). — _So sign is preserved: $-1$ lands in the past half (bucket 1), $+1$ in the future half (bucket 9)._
- Note that each bucket has its own learned per-head scalar, so "one token to my left" and "one token to my right" can be weighted differently. — _Direction-aware bias lets a head prefer, say, the previous token specifically._
- Connect to reverse: producing output position $k$ from input position $N{-}1{-}k$ requires the model to count from a known end and move in a consistent direction. — _If $+1$ and $-1$ shared a bucket, the model could not distinguish "the token before" from "the token after," and reversing would be impossible._

**Answer:** The bidirectional split sends negative offsets to the first 8 buckets and positive
                 offsets to the last 8, so offset $-1$ (bucket 1) and offset $+1$ (bucket 9) carry
                 different learned scalars. That direction-awareness is what lets a head treat "the token
                 to my left" differently from "the token to my right." Reversing a sequence depends on
                 exactly that distinction, which is why collapsing the two directions (or removing the
                 bias entirely) breaks <rev> while leaving the order-free
                 <last> task largely intact.

</details>

**Problem 3.** T5 pre-trains with span corruption before any task. For the sentence "Thank you for
            inviting me to your party last week.", suppose the tokens "for inviting" and "last" are the
            ones dropped. Write the corrupted input and the target in T5's sentinel format, and explain
            why this objective is itself "text-to-text."

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace each dropped contiguous span by one unique sentinel: "for inviting" &rarr; <X>, "last" &rarr; <Y>. — _\S 3.1.4: consecutive dropped tokens share a single sentinel; each span gets a distinct one._
- Write the input: "Thank you <X> me to your party <Y> week." — _The model sees the surviving tokens with sentinels marking the gaps._
- Write the target as the dropped spans, each prefixed by its sentinel, ending with a final sentinel: "<X> for inviting <Y> last <Z>". — _The decoder regenerates only the missing pieces, in order, delimited by sentinels._

**Answer:** Input: "Thank you <X> me to your party <Y> week."; target:
                 "<X> for inviting <Y> last <Z>" (Figure 2). It is
                 text-to-text because the model is given a string and asked to produce a string, using the
                 same encoder-decoder, vocabulary, and next-token loss as every downstream task —
                 so pre-training and fine-tuning differ only in the data, not the interface. That
                 uniformity is the whole point of T5.

</details>